# Task 8 — Mask-based baselines con EoMT

Notebook Colab per eseguire il task 8 sugli stessi dataset anomaly del task 7, leggendo dataset e pesi da Google Drive.

Flusso:
1. Monta Drive e prepara la repo.
2. Estrae `Anomaly_Validation_Datasets.zip` in `/content`.
3. Esegue EoMT COCO / Cityscapes / fine-tuned con MSP, MaxLogit, Entropy, RbA.
4. Esegue temperature scaling.
5. Legge il CSV finale dei risultati.


## 1. Setup ambiente, Drive e repository

In [ ]:
# Ambiente preparato da 00_setup (stesso server Colab): qui solo mount + path + sys.path.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "MaskArchitectureAnomaly_CourseProject"
EOMT_ROOT = PROJECT_ROOT / "eomt"
EVAL_DIR = PROJECT_ROOT / "eval"
LARGE_FILES = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project" / "large_files"
for p in (PROJECT_ROOT, EOMT_ROOT, EVAL_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print("EOMT_ROOT exists:", EOMT_ROOT.exists())


## 2. Path principali: repo, EoMT, dataset, pesi

In [ ]:
from pathlib import Path
import zipfile
import glob
import os

# Path dei dati/pesi su Drive (large_files).
WEIGHTS_ROOT = LARGE_FILES / "weights"
ANOMALY_ZIP = LARGE_FILES / "datasets" / "anomaly" / "Anomaly_Validation_Datasets.zip"

# Copia locale dei dataset (disco VM, piu' veloce dello zip su Drive)
LOCAL_DATA_DIR = Path("/content/anomaly_data")
DATA_ROOT = LOCAL_DATA_DIR / "Validation_Dataset"

# Output CSV su Drive (in 01_Project, accanto a large_files), sopravvive al reset del runtime
RESULTS_CSV = LARGE_FILES.parent / "results_task8_eomt_V4_with_stage0_model_class_n_mask_heads.csv"

EVAL_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EOMT_ROOT:", EOMT_ROOT, "| exists:", EOMT_ROOT.exists())
print("EVAL_DIR:", EVAL_DIR, "| exists:", EVAL_DIR.exists())
print("ANOMALY_ZIP:", ANOMALY_ZIP, "| exists:", ANOMALY_ZIP.exists())
print("WEIGHTS_ROOT:", WEIGHTS_ROOT, "| exists:", WEIGHTS_ROOT.exists())
print("RESULTS_CSV:", RESULTS_CSV)


## 3. Estrazione dataset anomaly in `/content`

Si estrae in `/content` per evitare di leggere continuamente immagini dallo zip/Drive durante l'inference.


In [21]:
print("Zip exists:", ANOMALY_ZIP.exists())

if not LOCAL_DATA_DIR.exists():
    print("Extracting zip temporarily to /content...")
    with zipfile.ZipFile(ANOMALY_ZIP, "r") as z:
        z.extractall(LOCAL_DATA_DIR)
    print("Done.")
else:
    print("Already extracted in this runtime.")

print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("Contenuto DATA_ROOT:")
for p in sorted(DATA_ROOT.iterdir()) if DATA_ROOT.exists() else []:
    print("-", p.name)


Zip exists: True
Already extracted in this runtime.
DATA_ROOT: /content/anomaly_data/Validation_Dataset
DATA_ROOT exists: True
Contenuto DATA_ROOT:
- .DS_Store
- FS_LostFound_full
- RoadAnomaly
- RoadAnomaly21
- RoadObsticle21
- fs_static


## 4. Definizione dataset anomaly e controllo numero immagini

In [22]:
datasets = {
    "FS_LostFound_full": DATA_ROOT / "FS_LostFound_full" / "images" / "*.png",
    "fs_static": DATA_ROOT / "fs_static" / "images" / "*.jpg",
    "RoadAnomaly": DATA_ROOT / "RoadAnomaly" / "images" / "*.jpg",
    "RoadAnomaly21": DATA_ROOT / "RoadAnomaly21" / "images" / "*.png",
    "RoadObsticle21": DATA_ROOT / "RoadObsticle21" / "images" / "*.webp",
}

for name, pattern in datasets.items():
    files = glob.glob(str(pattern))
    print(name, len(files), "images")


FS_LostFound_full 100 images
fs_static 30 images
RoadAnomaly 60 images
RoadAnomaly21 10 images
RoadObsticle21 30 images


## 5. Checkpoint EoMT

Aggiorna `EOMT_FINETUNED_WEIGHTS` se il tuo checkpoint fine-tuned ha un nome/path diverso.

I preset da usare sono:
- `coco` → `num_classes=133`, `num_q=200`
- `cityscapes` → `num_classes=19`, `num_q=100`
- `finetuned` → `num_classes=19`, `num_q=100`


In [23]:
EOMT_COCO_WEIGHTS = WEIGHTS_ROOT / "eomt_coco.bin"
EOMT_CITYSCAPES_WEIGHTS = WEIGHTS_ROOT / "eomt_cityscapes.bin"
checkpoints = {
    "eomt_coco": {
        "weights": EOMT_COCO_WEIGHTS,
        "preset": "coco",
    },
     "eomt_cityscapes": {
         "weights": EOMT_CITYSCAPES_WEIGHTS,
         "preset": "cityscapes",
     },
     "eomt_finetuned_stage0": {
         "weights": WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage0_heads" / "best.ckpt",
         "preset": "finetuned",
     },
     "eomt_finetuned_stage1": {
         "weights": WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage1_head" / "stage1_weights.bin",
         "preset": "finetuned",
     },
     "eomt_finetuned_stage2": {
         "weights": WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage2_unfreeze_last" / "stage2_weights.bin",
         "preset": "finetuned",
     },
}

for name, cfg in checkpoints.items():
    print(name, "->", cfg["weights"], "| exists:", cfg["weights"].exists())

eomt_coco -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/eomt_coco.bin | exists: True
eomt_cityscapes -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/eomt_cityscapes.bin | exists: True
eomt_finetuned_stage0 -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_stage0_heads/best.ckpt | exists: True
eomt_finetuned_stage1 -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_stage1_head/stage1_weights.bin | exists: True
eomt_finetuned_stage2 -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_stage2_unfreeze_last/stage2_weights.bin | exists: True


In [ ]:
# Verifica num_q / num_classes leggendoli direttamente dai checkpoint.
# num_q  = q.weight.shape[0]   (numero di mask queries imparate dal modello)
# num_classes = class_head.weight.shape[0] - 1   (-1 per la classe "no-object")
import torch

def inspect_ckpt(path):
    sd = torch.load(path, map_location="cpu")
    sd = sd["state_dict"] if isinstance(sd, dict) and "state_dict" in sd else sd
    num_q = num_classes = None
    for k, v in sd.items():
        kk = k.replace("._orig_mod", "").replace("network.", "").replace("module.", "")
        if kk.endswith("q.weight"):
            num_q = v.shape[0]
        if kk.endswith("class_head.weight"):
            num_classes = v.shape[0] - 1
    return num_q, num_classes

PRESET_EXPECTED = {"coco": (200, 133), "cityscapes": (100, 19), "finetuned": (200, 19)}

for name, cfg in checkpoints.items():
    if not cfg["weights"].exists():
        print(f"{name}: pesi non trovati"); continue
    q, c = inspect_ckpt(cfg["weights"])
    exp_q, exp_c = PRESET_EXPECTED[cfg["preset"]]
    ok = (q == exp_q) and (c == exp_c)
    flag = "OK" if ok else "!!! MISMATCH col preset"
    print(f"{name:28s} preset={cfg['preset']:11s} num_q={q} num_classes={c} "
          f"(atteso {exp_q}/{exp_c}) {flag}")


### 6.1 Pre-caricamento dei pesi dei checkpoint in memoria

Per evitare di ricaricare i pesi da disco più volte, carichiamo il `state_dict` di ogni checkpoint in memoria. Tuttavia, lo script `evalAnomaly_eomt.py` (eseguito tramite `subprocess`) caricherà comunque i pesi dal path fornito, poiché opera in un processo separato.

Per sfruttare questi pesi pre-caricati, lo script `evalAnomaly_eomt.py` dovrebbe essere modificato per accettare un `state_dict` già in memoria o per implementare una cache interna, oppure la logica di valutazione dovrebbe essere integrata direttamente in questo notebook.

In [ ]:
# import torch
# import sys

# total_weights_size_mb = 0
# print("--- Pre-caricamento pesi in memoria ---")

# for name, cfg in checkpoints.items():
#     if cfg["weights"].exists():
#         print(f"Caricamento pesi per {name}...")
#         # Carica il state_dict su CPU per non occupare memoria GPU finché non serve
#         loaded_state_dict = torch.load(cfg["weights"], map_location='cpu')
#         cfg["loaded_state_dict"] = loaded_state_dict

#         # Calcola la dimensione del state_dict in memoria
#         # Approssimazione: dimensione di ogni tensore * numero di elementi * dimensione in byte del tipo
#         # Questo è un valore stimato, il consumo reale potrebbe variare.
#         size_bytes = 0
#         for k, v in loaded_state_dict.items():
#             size_bytes += v.element_size() * v.nelement()
#         size_mb = size_bytes / (1024 * 1024)
#         total_weights_size_mb += size_mb
#         print(f"  -> Caricato. Dimensione approssimativa: {size_mb:.2f} MB")
#     else:
#         print(f"Skip {name}: pesi non trovati a {cfg['weights']}")

# print(f"--- Totale pesi pre-caricati in memoria: {total_weights_size_mb:.2f} MB ---")


## 6. Funzione per lanciare una singola evaluation

Uso `subprocess.run` con `PYTHONPATH` impostato in modo che lo script in `eval/` trovi i moduli EoMT dentro `eomt/models`.


In [24]:
# Path dello script di evaluation EoMT
SCRIPT_PATH = PROJECT_ROOT / "eval" / "evalAnomaly_eomt.py"

assert SCRIPT_PATH.exists(), f"Script non trovato: {SCRIPT_PATH}"

print("Script trovato:", SCRIPT_PATH)

Script trovato: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eval/evalAnomaly_eomt.py


In [ ]:
# =====================================================================
# Motore di valutazione IN-PROCESS (niente subprocess).
#
# Idea: per ogni checkpoint il modello si carica UNA volta, e per ogni
# dataset il forward si esegue UNA volta per immagine. I 4 metodi
# (msp/maxlogit/entropy/rba) e le varie temperature sono solo
# post-processing degli stessi logit -> niente forward ripetuti.
#
# Riusa le funzioni gia' testate di eval/evalAnomaly_eomt.py e la stessa
# save_csv -> il CSV in output e' identico a prima.
# =====================================================================
import sys, glob, os, time
from types import SimpleNamespace
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr

for p in [str(EOMT_ROOT), str(PROJECT_ROOT), str(EVAL_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import evalAnomaly_eomt as E  # build_eomt, load_eomt_weights, score fns, save_csv, transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = True  # autocast fp16 sul forward: ~2x piu' veloce. Metti False per match esatto coi run fp32.

# La temperatura cambia il risultato SOLO per questi metodi.
# maxlogit: invariante (scalatura monotona -> stesso ranking di AUPRC/FPR95); rba: ignora T.
TEMP_METHODS = {"msp", "entropy"}

print("Device:", DEVICE, "| AMP fp16:", USE_AMP)


def build_model(weights_path, preset):
    args = SimpleNamespace(
        preset=preset, num_blocks=3, patch_size=16,
        backbone_name="vit_base_patch14_reg4_dinov2",
    )
    args = E.apply_preset(args)
    model = E.build_eomt(args)
    t = time.time()
    print(f"  [load] {os.path.basename(str(weights_path))} ...", flush=True)
    model = E.load_eomt_weights(model, str(weights_path))
    print(f"  [load] completato in {time.time() - t:.1f}s", flush=True)
    return model.to(DEVICE).eval(), args


@torch.no_grad()
def eval_checkpoint(checkpoint_name, weights_path, preset, methods, temperatures=(1.0,)):
    """Carica il modello una volta e valuta tutti i dataset/metodi/temperature.

    Le temperature si applicano SOLO ai metodi in TEMP_METHODS (msp/entropy);
    maxlogit e rba sono valutati solo a T=1.0.
    """
    model, args = build_model(weights_path, preset)
    target_size = (E.IMG_HEIGHT, E.IMG_WIDTH)

    for dataset_name, pattern in datasets.items():
        paths = sorted(glob.glob(str(pattern)))
        if not paths:
            print(f"SKIP dataset vuoto: {dataset_name}")
            continue

        runs = [(m, T) for m in methods for T in (temperatures if m in TEMP_METHODS else (1.0,))]
        scores = {k: [] for k in runs}
        gts = {k: [] for k in runs}

        print(f"\n[{checkpoint_name}] {dataset_name}: {len(paths)} immagini")
        t_ds = time.time()
        for idx, path in enumerate(paths):
            img = E.input_transform(Image.open(path).convert("RGB")).unsqueeze(0).float().to(DEVICE)
            t_fwd = time.time()
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=(USE_AMP and DEVICE.type == "cuda")):
                ml_layers, cl_layers = model(img)
            ml, cl = ml_layers[-1].float(), cl_layers[-1].float()  # forward calcolato UNA volta
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            if idx == 0:
                # tempo del singolo forward (sync incluso) + tempo del post-processing successivo
                t_post = time.time()

            pathGT = E.get_gt_path(path)
            if not os.path.exists(pathGT):
                continue
            gt = E.convert_gt(np.array(E.target_transform(Image.open(pathGT))), pathGT)
            if 1 not in np.unique(gt):
                continue
            valid = (gt == 0) | (gt == 1)
            gt_v = gt[valid].astype(np.uint8)

            pix = E.eomt_to_pixel_logits(ml, cl, target_size)  # condiviso da msp/maxlogit/entropy
            for (m, T) in runs:
                if m == "rba":
                    s = E.compute_rba_score(ml, cl, target_size)
                else:
                    s = E.compute_anomaly_score(pix, m, T)
                s = s.squeeze(0).float().cpu().numpy()[valid].astype(np.float32)
                scores[(m, T)].append(s)
                gts[(m, T)].append(gt_v)

            if idx == 0:
                print(f"  [forward/img] ~{t_post - t_fwd:.2f}s | [post+metodi/img] ~{time.time() - t_post:.2f}s", flush=True)

        print(f"  [dataset] {dataset_name} processato in {time.time() - t_ds:.1f}s", flush=True)

        for (m, T) in runs:
            if not gts[(m, T)]:
                continue
            label = np.concatenate(gts[(m, T)])
            out = np.concatenate(scores[(m, T)])
            auprc = average_precision_score(label, out)
            fpr95 = fpr_at_95_tpr(out, label)

            # riempie args e riusa la save_csv originale -> stesse colonne del CSV
            args.checkpoint_name = checkpoint_name
            args.weights = str(weights_path)
            args.input = [str(pattern)]
            args.method = m
            args.temperature = T
            args.results_csv = str(RESULTS_CSV)
            E.save_csv(
                args, auprc, fpr95,
                num_images=len(gts[(m, T)]),
                num_pixels=len(label),
                num_anomaly_pixels=int(label.sum()),
            )
            print(f"  {m:8s} T={T:<4} -> AUPRC={auprc*100:.2f}  FPR95={fpr95*100:.2f}")

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


## 7. Test su un solo dataset/metodo

Prima di lanciare tutto, fai un test piccolo. Consiglio: `RoadAnomaly` + `eomt_cityscapes` + `msp`.


In [14]:
RUN_SMOKE_TEST = False # Impostato a False per saltare il test rapido

## 8. Run completo Task 8: 3 checkpoint × 5 dataset × 4 metodi

Lascia `RUN_FULL = False` finché lo smoke test non funziona.  
Poi metti `RUN_FULL = True`.

Metodi:
- `msp`
- `maxlogit`
- `entropy`
- `rba`


In [ ]:
RUN_FULL = True

methods = ["msp", "maxlogit", "entropy", "rba"]

# Temperature scaling: applicato a msp ed entropy (maxlogit e' invariante, rba ignora T).
# Tutto in UNA passata: i forward sono condivisi, le T sono solo post-processing -> costo ~zero.
temperatures = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]

if RUN_FULL:
    for checkpoint_name, cfg in checkpoints.items():
        if not cfg["weights"].exists():
            print(f"Skip {checkpoint_name}: weights non trovati")
            continue
        # 1 load del modello + 1 forward per immagine; tutti i metodi e tutte le T dai logit in cache
        eval_checkpoint(
            checkpoint_name=checkpoint_name,
            weights_path=cfg["weights"],
            preset=cfg["preset"],
            methods=methods,
            temperatures=temperatures,
        )
else:
    print("RUN_FULL è False. Cambialo a True per lanciare il run completo.")


## 9. Temperature scaling

Il task chiede di provare temperature scaling.  
Qui lo applichiamo a `MSP`.

Puoi scegliere checkpoint e dataset. Per il run completo, puoi ciclare su tutti i dataset.


In [ ]:
# Il temperature scaling e' ora prodotto direttamente dal run principale (cella 8):
# msp ed entropy vengono valutati a tutte le `temperatures` per TUTTI i checkpoint,
# nella stessa passata. Quindi di norma NON serve eseguire questa cella.
#
# Usala solo se vuoi provare temperature EXTRA su un singolo checkpoint senza
# rifare tutti i modelli (rifa' comunque il forward per quel checkpoint).
RUN_EXTRA_TEMPERATURE = False

EXTRA_CHECKPOINT_NAME = "eomt_cityscapes"
EXTRA_TEMPERATURES = [0.01, 0.05, 5.0, 10.0]   # temperature aggiuntive da esplorare

if RUN_EXTRA_TEMPERATURE:
    cfg = checkpoints[EXTRA_CHECKPOINT_NAME]
    eval_checkpoint(
        checkpoint_name=EXTRA_CHECKPOINT_NAME,
        weights_path=cfg["weights"],
        preset=cfg["preset"],
        methods=["msp", "entropy"],
        temperatures=EXTRA_TEMPERATURES,
    )
else:
    print("Temperature scaling gia' incluso nel run principale (cella 8).")
    print("Imposta RUN_EXTRA_TEMPERATURE = True solo per temperature extra su un singolo checkpoint.")


## 10. Lettura risultati CSV

In [ ]:
import pandas as pd

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    display(df.tail(30))
else:
    print("CSV non ancora creato:", RESULTS_CSV)


,checkpoint_name,preset,weights,input,method,temperature,num_classes,num_q,num_blocks,img_height,img_width,AUPRC,FPR95,num_images,num_pixels,num_anomaly_pixels
123,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,entropy,1.0,19,200,3,1024,2048,55.649452,25.921412,60,125829120,12393891
124,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,rba,1.0,19,200,3,1024,2048,53.895970,31.630472,60,125829120,12393891
125,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,msp,1.0,19,200,3,1024,2048,71.524548,84.522276,10,20241275,2996730
126,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,maxlogit,1.0,19,200,3,1024,2048,70.728895,84.564539,10,20241275,2996730
127,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,entropy,1.0,19,200,3,1024,2048,67.686054,84.120458,10,20241275,2996730
128,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,rba,1.0,19,200,3,1024,2048,69.779147,30.690436,10,20241275,2996730
129,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadO...,msp,1.0,19,200,3,1024,2048,80.005401,2.137239,30,19454317,131306
130,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadO...,maxlogit,1.0,19,200,3,1024,2048,78.624578,2.134931,30,19454317,131306
131,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadO...,entropy,1.0,19,200,3,1024,2048,78.756167,2.119778,30,19454317,131306
132,eomt_finetuned_stage1,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadO...,rba,1.0,19,200,3,1024,2048,87.571962,2.106028,30,19454317,131306


## 11. Tabella pivot per report

Questa cella crea una tabella leggibile con righe `checkpoint/method/temperature` e colonne per dataset.


In [ ]:
if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    # Ricava nome dataset dal path input
    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in datasets.keys():
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)

    pivot_auprc = df.pivot_table(
        index=["checkpoint_name", "preset", "method", "temperature"],
        columns="dataset",
        values="AUPRC",
        aggfunc="last",
    )

    pivot_fpr95 = df.pivot_table(
        index=["checkpoint_name", "preset", "method", "temperature"],
        columns="dataset",
        values="FPR95",
        aggfunc="last",
    )

    print("AUPRC")
    display(pivot_auprc)

    print("FPR95")
    display(pivot_fpr95)
else:
    print("CSV non ancora creato.")


AUPRC


dataset                                                FS_LostFound_full  \
checkpoint_name       preset     method   temperature                      
eomt_cityscapes       cityscapes entropy  1.0                  21.485129   
                                 maxlogit 1.0                  22.205279   
                                 msp      1.0                  22.315284   
                                 rba      1.0                  22.163063   
eomt_coco             coco       entropy  1.0                   1.467183   
                                 maxlogit 1.0                   1.721693   
                                 msp      1.0                   1.733166   
                                 rba      1.0                   2.417288   
eomt_finetuned        finetuned  entropy  1.0                  36.057659   
                                 maxlogit 1.0                  35.192410   
                                 msp      1.0                  34.699996   
                                 rba      1.0                  28.168885   
eomt_finetuned_stage0 finetuned  entropy  1.0                  36.107171   
                                 maxlogit 1.0                  35.242350   
                                 msp      1.0                  34.750334   
                                 rba      1.0                  28.217550   
eomt_finetuned_stage1 finetuned  entropy  1.0                  44.128479   
                                 maxlogit 1.0                  43.957109   
                                 msp      1.0                  44.045389   
                                 rba      1.0                  41.094896   
eomt_finetuned_stage2 finetuned  entropy  1.0                  48.044795   
                                 maxlogit 1.0                  47.656850   
                                 msp      1.0                  47.557736   
                                 rba      1.0                  46.577055   

dataset                                                RoadAnomaly  \
checkpoint_name       preset     method   temperature                
eomt_cityscapes       cityscapes entropy  1.0            73.796041   
                                 maxlogit 1.0            71.567064   
                                 msp      1.0            72.403950   
                                 rba      1.0            72.929604   
eomt_coco             coco       entropy  1.0            47.123714   
                                 maxlogit 1.0            44.876668   
                                 msp      1.0            44.797527   
                                 rba      1.0            39.247326   
eomt_finetuned        finetuned  entropy  1.0            61.749363   
                                 maxlogit 1.0            62.951018   
                                 msp      1.0            64.769487   
                                 rba      1.0            64.790018   
eomt_finetuned_stage0 finetuned  entropy  1.0            61.780918   
                                 maxlogit 1.0            62.977420   
                                 msp      1.0            64.807760   
                                 rba      1.0            64.787080   
eomt_finetuned_stage1 finetuned  entropy  1.0            67.686054   
                                 maxlogit 1.0            70.728895   
                                 msp      1.0            71.524548   
                                 rba      1.0            69.779147   
eomt_finetuned_stage2 finetuned  entropy  1.0            56.198828   
                                 maxlogit 1.0            64.359378   
                                 msp      1.0            67.345286   
                                 rba      1.0            64.680266   

dataset                                                RoadObsticle21  \
checkpoint_name       preset     method   temperature                   
eomt_cityscapes       cityscapes entropy  1.0           

FPR95


dataset                                                FS_LostFound_full  \
checkpoint_name       preset     method   temperature                      
eomt_cityscapes       cityscapes entropy  1.0                  19.670153   
                                 maxlogit 1.0                  19.575393   
                                 msp      1.0                  19.790849   
                                 rba      1.0                  19.725290   
eomt_coco             coco       entropy  1.0                  90.008073   
                                 maxlogit 1.0                  91.439397   
                                 msp      1.0                  91.466357   
                                 rba      1.0                  41.620494   
eomt_finetuned        finetuned  entropy  1.0                  45.881843   
                                 maxlogit 1.0                  46.562407   
                                 msp      1.0                  27.911716   
                                 rba      1.0                  35.319529   
eomt_finetuned_stage0 finetuned  entropy  1.0                  45.878735   
                                 maxlogit 1.0                  46.532467   
                                 msp      1.0                  27.876142   
                                 rba      1.0                  35.300111   
eomt_finetuned_stage1 finetuned  entropy  1.0                  20.420171   
                                 maxlogit 1.0                  21.467471   
                                 msp      1.0                  19.817449   
                                 rba      1.0                  31.351560   
eomt_finetuned_stage2 finetuned  entropy  1.0                  21.300697   
                                 maxlogit 1.0                  22.393155   
                                 msp      1.0                  21.740548   
                                 rba      1.0                  31.075988   

dataset                                                RoadAnomaly  \
checkpoint_name       preset     method   temperature                
eomt_cityscapes       cityscapes entropy  1.0            25.080625   
                                 maxlogit 1.0            25.770028   
                                 msp      1.0            24.990741   
                                 rba      1.0            22.330976   
eomt_coco             coco       entropy  1.0            83.386769   
                                 maxlogit 1.0            89.718627   
                                 msp      1.0            89.827565   
                                 rba      1.0            91.582690   
eomt_finetuned        finetuned  entropy  1.0            58.910371   
                                 maxlogit 1.0            64.508362   
                                 msp      1.0            56.243374   
                                 rba      1.0            13.615645   
eomt_finetuned_stage0 finetuned  entropy  1.0            58.755578   
                                 maxlogit 1.0            64.291009   
                                 msp      1.0            55.888088   
                                 rba      1.0            13.604279   
eomt_finetuned_stage1 finetuned  entropy  1.0            84.120458   
                                 maxlogit 1.0            84.564539   
                                 msp      1.0            84.522276   
                                 rba      1.0            30.690436   
eomt_finetuned_stage2 finetuned  entropy  1.0            77.135159   
                                 maxlogit 1.0            56.065875   
                                 msp      1.0            35.338313   
                                 rba      1.0            31.769429   

dataset                                                RoadObsticle21  \
checkpoint_name       preset     method   temperature                   
eomt_cityscapes       cityscapes entropy  1.0           

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 1.7 MB/s eta 0:00:00a 0:00:01


In [ ]:
# Excel: TABELLA UNICA stile report.
# Righe = (Model = checkpoint, Method); colonne multi-livello = (dataset, AuPRC/FPR95).
# Solo temperature == 1.0 (niente temperature scaling).
import pandas as pd

# Dataset -> intestazione del report, NELL'ORDINE voluto
DATASET_COLS = {
    "RoadObsticle21": "SMIYC RO-21",
    "FS_LostFound_full": "FS L&F",
    "fs_static": "FS Static",
    "RoadAnomaly": "Road Anomaly",
}
# checkpoint_name -> etichetta Model
MODEL_LABELS = {
    "eomt_coco": "EoMT COCO",
    "eomt_cityscapes": "EoMT Cityscapes",
    "eomt_finetuned_stage0": "EoMT ft stage0",
    "eomt_finetuned_stage1": "EoMT ft stage1",
    "eomt_finetuned_stage2": "EoMT ft stage2",
}
# metodo CSV -> etichetta, nell'ordine
METHOD_LABELS = {"msp": "MSP", "maxlogit": "MaxLogit", "entropy": "Max Entropy", "rba": "RbA"}

if RESULTS_CSV.exists():
    EXCEL_OUTPUT_PATH = RESULTS_CSV.parent / "notebook_colab/task_8/task8_results.xlsx"
    df = pd.read_csv(RESULTS_CSV)

    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in datasets.keys():
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)
    df = df[df["temperature"] == 1.0]

    col_tuples = [(lbl, metric) for lbl in DATASET_COLS.values() for metric in ("AuPRC", "FPR95")]
    data = {ct: [] for ct in col_tuples}
    index_tuples = []

    for ckpt in checkpoints.keys():            # ordine checkpoint
        for m in METHOD_LABELS:                # ordine metodi
            sub = df[(df["checkpoint_name"] == ckpt) & (df["method"] == m)]
            if sub.empty:
                continue
            index_tuples.append((MODEL_LABELS.get(ckpt, ckpt), METHOD_LABELS[m]))
            for ds_raw, ds_label in DATASET_COLS.items():
                r = sub[sub["dataset"] == ds_raw]
                a = round(float(r["AUPRC"].iloc[-1]), 2) if not r.empty else None
                f = round(float(r["FPR95"].iloc[-1]), 2) if not r.empty else None
                data[(ds_label, "AuPRC")].append(a)
                data[(ds_label, "FPR95")].append(f)

    table = pd.DataFrame(
        data,
        index=pd.MultiIndex.from_tuples(index_tuples, names=["Model", "Method"]),
    )
    table.columns = pd.MultiIndex.from_tuples(table.columns)

    with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine="xlsxwriter") as writer:
        table.to_excel(writer, sheet_name="Task8")

    print("Excel salvato:", EXCEL_OUTPUT_PATH)
    display(table)
else:
    print("CSV non ancora creato. Impossibile salvare in Excel.")


Excel salvato: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/notebook_colab/task_8/task8_results.xlsx


SMIYC RO-21         FS L&F        FS Static  \
                                  AuPRC   FPR95  AuPRC  FPR95     AuPRC   
Model           Method                                                    
EoMT COCO       MSP                2.36   99.99   1.73  91.47     13.87   
                MaxLogit           2.41   99.99   1.72  91.44     13.73   
                Max Entropy        5.30  100.00   1.47  90.01     13.02   
                RbA                2.47  100.00   2.42  41.62     13.38   
EoMT Cityscapes MSP               94.72    0.31  22.32  19.79     40.65   
                MaxLogit          94.72    0.32  22.21  19.58     40.75   
                Max Entropy       94.69    0.32  21.49  19.67     38.79   
                RbA               94.77    0.31  22.16  19.73     40.61   
EoMT ft stage0  MSP               42.49    4.01  34.75  27.88     57.93   
                MaxLogit          40.06    4.11  35.24  46.53     58.88   
                Max Entropy       39.22    4.20  36.11  45.88     60.80   
                RbA               49.08    3.78  28.22  35.30     60.56   
EoMT ft stage1  MSP               80.01    2.14  44.05  19.82     72.97   
                MaxLogit          78.62    2.13  43.96  21.47     73.15   
                Max Entropy       78.76    2.12  44.13  20.42     73.73   
                RbA               87.57    2.11  41.09  31.35     73.55   
EoMT ft stage2  MSP               77.78    2.18  47.56  21.74     73.08   
                MaxLogit          76.37    2.17  47.66  22.39     73.65   
                Max Entropy       75.35    2.15  48.04  21.30     74.66   
                RbA               82.65    2.14  46.58  31.08     72.40   

                                   Road Anomaly         
                             FPR95        AuPRC  FPR95  
Model           Method                                  
EoMT COCO       MSP          97.67        44.80  89.83  
                MaxLogit     97.58        44.88  89.72  
                Max Entropy  96.84        47.12  83.39  
                RbA          99.99        39.25  91.58  
EoMT Cityscapes MSP          61.16        72.40  24.99  
                MaxLogit     61.64        71.57  25.77  
                Max Entropy  61.11        73.80  25.08  
                RbA          56.12        72.93  22.33  
EoMT ft stage0  MSP          86.03        64.81  55.89  
                MaxLogit     86.03        62.98  64.29  
                Max Entropy  85.70        61.78  58.76  
                RbA          28.39        64.79  13.60  
EoMT ft stage1  MSP          23.26        71.52  84.52  
                MaxLogit     22.50        70.73  84.56  
                Max Entropy  22.28        67.69  84.12  
                RbA          32.41        69.78  30.69  
EoMT ft stage2  MSP          26.51        67.35  35.34  
                MaxLogit     25.76        64.36  56.07  
                Max Entropy  25.70        56.20  77.14  
                RbA          32.27        64.68  31.77

In [ ]:
# Excel TEMPERATURE SCALING: un foglio per checkpoint.
# Per ogni metodo con piu' temperature (msp, entropy): righe MSP / MSP(t=...) / MSP (best t),
# poi Max Entropy / ... . Colonne = mIoU (placeholder) + (dataset, AuPRC/FPR95).
# "best t" = per cella: AuPRC alla T che la massimizza, FPR95 alla T che lo minimizza.
import pandas as pd

DATASET_COLS = {
    "RoadObsticle21": "SMIYC RO-21",
    "FS_LostFound_full": "FS L&F",
    "fs_static": "FS Static",
    "RoadAnomaly": "Road Anomaly",
}
TEMP_METHOD_LABELS = {"msp": "MSP", "entropy": "Max Entropy"}  # metodi su cui T ha effetto

if RESULTS_CSV.exists():
    TEMP_EXCEL_PATH = RESULTS_CSV.parent / "notebook_colab/task_8/task8_temperature_results.xlsx"
    df = pd.read_csv(RESULTS_CSV)

    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in datasets.keys():
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)

    col_tuples = [("", "mIoU")] + [(lbl, met) for lbl in DATASET_COLS.values()
                                   for met in ("AuPRC", "FPR95")]

    def vals_at_temp(g, T, ds_raw):
        r = g[(g["dataset"] == ds_raw) & (g["temperature"] == T)]
        if r.empty:
            return None, None
        return round(float(r["AUPRC"].iloc[-1]), 2), round(float(r["FPR95"].iloc[-1]), 2)

    def vals_best(g, ds_raw):  # best per cella, per metrica
        r = g[g["dataset"] == ds_raw]
        if r.empty:
            return None, None
        return round(float(r["AUPRC"].max()), 2), round(float(r["FPR95"].min()), 2)

    sheets = {}
    for ckpt in checkpoints.keys():
        data = {ct: [] for ct in col_tuples}
        index_labels = []
        for m, mlabel in TEMP_METHOD_LABELS.items():
            g = df[(df["checkpoint_name"] == ckpt) & (df["method"] == m)]
            if g["temperature"].nunique() <= 1:
                continue
            temps = sorted(g["temperature"].unique())
            rows = []
            if 1.0 in temps:
                rows.append((mlabel, lambda ds, g=g: vals_at_temp(g, 1.0, ds)))
            for T in [t for t in temps if t != 1.0]:
                rows.append((f"{mlabel}(t={T})", lambda ds, g=g, T=T: vals_at_temp(g, T, ds)))
            rows.append((f"{mlabel} (best t)", lambda ds, g=g: vals_best(g, ds)))
            for label, getter in rows:
                index_labels.append(label)
                data[("", "mIoU")].append("----")
                for ds_raw, ds_label in DATASET_COLS.items():
                    a, f = getter(ds_raw)
                    data[(ds_label, "AuPRC")].append(a)
                    data[(ds_label, "FPR95")].append(f)
        if index_labels:
            table = pd.DataFrame(data, index=pd.Index(index_labels, name="Method"))
            table.columns = pd.MultiIndex.from_tuples(table.columns)
            sheets[ckpt] = table

    if not sheets:
        print("Nessun temperature scaling trovato (serve >1 temperatura per msp/entropy). Esegui la cella 8.")
    else:
        with pd.ExcelWriter(TEMP_EXCEL_PATH, engine="xlsxwriter") as writer:
            for ckpt, table in sheets.items():
                table.to_excel(writer, sheet_name=ckpt[:31])
        print("Excel temperature salvato:", TEMP_EXCEL_PATH)
        print("Fogli (un checkpoint ciascuno):", list(sheets.keys()))
        display(list(sheets.values())[-1])  # anteprima dell'ultimo checkpoint
else:
    print("CSV non ancora creato.")
